In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, mean_absolute_error, r2_score, mean_squared_error
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
import re

In [2]:
#Import dataset
df_melbourne = pd.read_csv('Melbourne_housing.csv', low_memory=False)

df_california = pd.read_csv("california_housing_data.csv")

df_cars = pd.read_csv("Cars Datasets 2025.csv", encoding='latin-1')


**DATA CLEANING FOR MELBOURNE HOUSING**

In [3]:
columns_to_use = ['Suburb', 'Rooms', 'Type', 'Method', 'SellerG', 'Regionname', 'Propertycount', 'Distance', 'Bedroom', 'Bathroom', 'Postcode','Car', 'Price']
df_melbourne = df_melbourne[columns_to_use]

columns_to_fill_0 = ['Car', 'Bathroom', 'Bedroom']
df_melbourne[columns_to_fill_0] = df_melbourne[columns_to_fill_0].fillna(0)

df_melbourne = df_melbourne.replace([np.inf, -np.inf], np.nan).dropna()

df_melbourne = pd.get_dummies(df_melbourne, drop_first=True)

**DATA CLEANING CARS**

In [4]:
# Drop unnecessary columns
df_cars = df_cars.drop(columns=['Engines', 'Fuel Types'])

# Normalize the brand name
def normalize_brand(name):
    name = str(name).strip().lower()           # Remove extra spaces and lowercase
    name = re.sub(r"\s+", " ", name)          # Replace multiple spaces with a single space

    if name in ["bmw", "gmc"]:
        return name.upper()                    # Keep certain brands in uppercase

    elif "rolls" in name:
        return "Rolls-Royce"

    elif "mercedes" in name:
        return "Mercedes-Benz"

    return name.title()                        # Capitalize each word for other brands

df_cars["Company Names"] = df_cars["Company Names"].apply(normalize_brand)

# Count the number of rows per company
df_cars.groupby("Company Names").count()

# Normalize seat information
def normalize_seats(x):
    x = str(x).strip().lower()
    x = re.sub(r"\s+", ",", x)                # Replace spaces with commas

    if "+" in x:                              # If it contains a '+', sum all numbers found
        nums = re.findall(r"\d+", x)
        if nums:
            return sum(int(n) for n in nums)
    nums = re.findall(r"\d+", x)
    if len(nums) >= 2:                        # If more than one number, return the average
        nums = [int(n) for n in nums]
        return int(sum(nums)/len(nums))
    else:
        return x                              # Otherwise, leave as-is

df_cars['Seats'] = df_cars["Seats"].apply(normalize_seats)

# Convert string values to numeric
def extract_numeric(x):
    if pd.isna(x):
        return np.nan
    s = str(x)
    m = re.findall(r"[-+]?\d*\.?\d+", s)      # Extract numbers including decimals
    if not m:
        return np.nan
    return float(m[0])

numeric_cols_raw = [
    "CC/Battery Capacity",
    "HorsePower",
    "Total Speed",
    "Performance(0 - 100 )KM/H",
    "Seats",
    "Cars Prices",
    "Torque"
]

# Apply numeric extraction to all relevant columns
for col in numeric_cols_raw:
    if col in df_cars.columns:
        df_cars[col] = df_cars[col].apply(extract_numeric)

# List numeric columns
numeric_cols = df_cars.select_dtypes(include=["float", "int"]).columns.tolist()

# Fill missing numeric values with the column mean (lightweight handling)
df_cars[numeric_cols].fillna(df_cars[numeric_cols].mean())
df_cars[numeric_cols].dropna()               # Also drop remaining NaNs

# Convert categorical columns to dummy variables
df_cars = pd.get_dummies(df_cars, drop_first=True)

**OUTLIERS**

In [5]:
#remove outlier
numeric_cols = df_melbourne.select_dtypes(include=["float", "int"]).columns.tolist()

for col in numeric_cols:
    Q1 = df_melbourne[col].quantile(0.2)
    Q3 = df_melbourne[col].quantile(0.8)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

    df_melbourne[col] = np.where(df_melbourne[col].between(lower, upper), df_melbourne[col], np.nan)

df_melbourne = df_melbourne.dropna()

In [6]:
#remove outlier
numeric_cols = df_california.select_dtypes(include=["float", "int"]).columns.tolist()

for col in numeric_cols:
    Q1 = df_california[col].quantile(0.2)
    Q3 = df_california[col].quantile(0.8)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

    df_california[col] = np.where(df_california[col].between(lower, upper), df_california[col], np.nan)

df_california = df_california.dropna()

In [7]:
#remove outlier
numeric_cols = df_cars.select_dtypes(include=["float", "int"]).columns.tolist()

for col in numeric_cols:
    Q1 = df_cars[col].quantile(0.2)
    Q3 = df_cars[col].quantile(0.8)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

    df_cars[col] = np.where(df_cars[col].between(lower, upper), df_cars[col], np.nan)

df_cars = df_cars.dropna()

**HYPERPARAMETER DISTRIBUTION FOR GRID SEARCH**

In [8]:
param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],   
    'max_depth': [3, 5],             
    'subsample': [0.8, 1.0], 
    'eval_metric': ['logloss', 'rmse'],  
    'sampling_method' : ['uniform', 'gradient_based'],      
    'subsample': [0.25, 0.5],
}

**MELBOURNE HOUSING ML**

In [9]:
# Split features and target
X = df_melbourne.drop('Price', axis='columns')  # All columns except 'Price'
y = df_melbourne["Price"]                       # Target variable

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features to zero mean and unit variance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define an XGBoost regressor
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',  # Standard regression objective
    use_label_encoder=False,        # Avoid warning for label encoding
    random_state=42,
    n_jobs=3                        # Use 3 CPU cores
)

# Set up grid search for hyperparameter tuning
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,          # Dictionary of hyperparameters to try
    cv=5,                           # 5-fold cross-validation
    scoring='neg_mean_squared_error',  # Evaluate using MSE (negative)
    verbose=1,                       # Print progress
    n_jobs=3                          # Parallelize across 3 cores
)

print("\n--- Starting Hyperparameter Tuning (Please wait) ---")
grid_search.fit(X_train, y_train)  # Run the grid search

print("\n--- Tuning Complete ---")
print(f"Best Parameters Found: {grid_search.best_params_}")

# Convert negative MSE to RMSE
best_rmse = np.sqrt(-grid_search.best_score_)
print(f"Best RMSE (CV mean): {best_rmse:.4f}")

# Train the best model on training data
best_xgb = grid_search.best_estimator_

# Predict on test set
y_pred = best_xgb.predict(X_test)

print("\n--- Final Report (Test Set) ---")
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE Test Set: {rmse_test:.4f}")

r2 = r2_score(y_test, y_pred)
print(f"R2 Test Set: {r2:.4f}")


--- Starting Hyperparameter Tuning (Please wait) ---
Fitting 5 folds for each of 64 candidates, totalling 320 fits


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [22:47:21] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [22:47:21] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [22:47:21] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [22:47:22] WARN


--- Tuning Complete ---
Best Parameters Found: {'eval_metric': 'logloss', 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'sampling_method': 'uniform', 'subsample': 0.5}
Best RMSE (CV mean): 225344.4900

--- Final Report (Test Set) ---
RMSE Test Set: 225586.5098
R2 Test Set: 0.7756


**CALIFORNIA HOUSING ML**

In [10]:
# Separate features and target
X = df_california.drop(columns=["MedHouseVal"])  # X contains all columns except the target
y = df_california["MedHouseVal"]                # y is the target variable (median house value)

# Split data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features to have mean 0 and variance 1
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define XGBoost regressor
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',   # Regression using squared error
    eval_metric='logloss',           # Evaluation metric
    use_label_encoder=False,
    random_state=42,
    n_jobs=3                         # Use 3 CPU threads
)

# Set up grid search for hyperparameter tuning
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,            # Dictionary of parameters to try
    cv=5,                             # 5-fold cross-validation
    scoring='neg_mean_squared_error', # Metric to optimize (negative MSE)
    verbose=1,                        # Print progress
    n_jobs=3                          # Parallelize with 3 CPU threads
)

print("\n--- Starting Tuning (Please Wait...) ---")
grid_search.fit(X_train, y_train)     # Run the hyperparameter search

print("\n--- Tuning Completed ---")
print(f"Best Parameters Found: {grid_search.best_params_}")
best_rmse = np.sqrt(-grid_search.best_score_)
print(f"Best RMSE (Cross-Validation Mean): {best_rmse:.4f}")

# Get the best model and predict on the test set
best_xgb = grid_search.best_estimator_
y_pred = best_xgb.predict(X_test)

print("\n--- Final Report (Test Set) ---")
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Test Set RMSE: {rmse_test:.4f}")

r2 = r2_score(y_test, y_pred)
print(f"Test Set R2: {r2:.4f}")


--- Starting Tuning (Please Wait...) ---
Fitting 5 folds for each of 64 candidates, totalling 320 fits


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [22:48:46] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [22:48:46] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [22:48:46] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [22:48:46] WARN


--- Tuning Completed ---
Best Parameters Found: {'eval_metric': 'logloss', 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'sampling_method': 'uniform', 'subsample': 0.5}
Best RMSE (Cross-Validation Mean): 0.4572

--- Final Report (Test Set) ---
Test Set RMSE: 0.4639
Test Set R2: 0.8219


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
160 fits failed out of a total of 320.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
20 fits failed with the following error:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/core.py", line 774, in inner_f
    return func(**kwargs)
  File "/Library/Frameworks/Pyt

**CARS PARAM ML**

In [11]:
# Separate features and target
X = df_cars.drop('Cars Prices', axis='columns')  # X contains all columns except the target
y = df_cars['Cars Prices']                       # y is the target variable (car prices)

# Split data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define XGBoost regressor
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',   # Regression using squared error
    eval_metric='logloss',           # Evaluation metric (note: unusual for regression)
    use_label_encoder=False,
    random_state=42,
    n_jobs=3                         # Use 3 CPU threads
)

# Set up grid search for hyperparameter tuning
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,            # Dictionary of parameters to test
    cv=5,                             # 5-fold cross-validation
    scoring='neg_mean_squared_error', # Metric to optimize (negative MSE)
    verbose=1,                        # Print progress
    n_jobs=3                          # Parallelize with 3 CPU threads
)

print("\n--- Starting Tuning (Please Wait...) ---")
grid_search.fit(X_train, y_train)     # Run the hyperparameter search

print("\n--- Tuning Completed ---")
print(f"Best Parameters Found: {grid_search.best_params_}")
best_rmse = np.sqrt(-grid_search.best_score_)
print(f"Best RMSE (Cross-Validation Mean): {best_rmse:.4f}")

# Predict on the test set using the best model
best_xgb = grid_search.best_estimator_
y_pred = best_xgb.predict(X_test)

print("\n--- Final Report (Test Set) ---")
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Test Set RMSE: {rmse_test:.4f}")

r2 = r2_score(y_test, y_pred)
print(f"Test Set R2: {r2:.4f}")


--- Starting Tuning (Please Wait...) ---
Fitting 5 folds for each of 64 candidates, totalling 320 fits


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [22:48:55] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [22:48:55] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [22:48:55] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [22:48:56] WARN


--- Tuning Completed ---
Best Parameters Found: {'eval_metric': 'logloss', 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'sampling_method': 'uniform', 'subsample': 0.5}
Best RMSE (Cross-Validation Mean): 9.3451

--- Final Report (Test Set) ---
Test Set RMSE: 10.4626
Test Set R2: 0.8583
